# Fire Incidence in Montana Rangeland & Grassland

Analyzes where wildfire and prescribed fire have occurred within rangeland and grassland land cover in Montana, using LCMS Change, LCMS Land Cover/Use, and MTBS burn severity (1985–2024).

**Key classes used**
| Product | Class | Value |
|---------|-------|-------|
| Change | Wildfire | 7 |
| Change | Prescribed Fire | 6 |
| Land Cover | Shrubs | 7 |
| Land Cover | Grass/Forb/Herb & Shrubs Mix | 8 |
| Land Cover | Grass/Forb/Herb | 10 |
| Land Use | Rangeland or Pasture | 5 |

In [ ]:
import os, ee
from IPython.display import display, HTML

# ee.Initialize MUST come before geeViz imports
EE_PROJECT = 'rcr-gee'
ee.Initialize(project=EE_PROJECT)

# Keep geeViz HTTP server in-process (prevents stale-layer bug)
os.environ['GEEVIZ_EEAUTH_MODE'] = 'auto'

import geeViz.getImagesLib as gil
import geeViz.geeView
from geeViz.outputLib import charts as cl

Map = gil.Map
Map.port = 8080
Map.project = EE_PROJECT
Map.clearMap()

print('EE initialized:', ee.Image(1).getInfo())


In [ ]:
START_YEAR = 1985
END_YEAR   = 2024

# Montana state boundary (TIGER)
montana = (
    ee.FeatureCollection('TIGER/2018/States')
    .filter(ee.Filter.eq('NAME', 'Montana'))
)
aoi = montana.geometry()

# UTM zone 12N — covers most of Montana
CRS = 'EPSG:32612'

# LCMS v2024-10 (most recent stable release)
LCMS_ASSET = 'projects/gtac-data-publish/assets/LCMS/Product_Version/2025-11'
lcms = (
    ee.ImageCollection(LCMS_ASSET)
    .filter(ee.Filter.bounds(aoi))
    .filter(ee.Filter.calendarRange(START_YEAR, END_YEAR, 'year'))
)

# MTBS annual burn severity mosaics
mtbs = (
    ee.ImageCollection('USFS/GTAC/MTBS/annual_burn_severity_mosaics/v1')
    .select([0], ['Severity'])
    .filter(ee.Filter.calendarRange(START_YEAR, END_YEAR, 'year'))
)

print(f'LCMS image count: {lcms.size().getInfo()}')
print(f'MTBS image count: {mtbs.size().getInfo()}')
print(f'Montana area: {aoi.area(500).divide(1e9).getInfo():.0f} thousand km²')


## 1 · Rangeland/Grassland Fire Map

The map shows four layers:
- **Wildfire pixels** (LCMS Change class 7) — any year where wildfire was the dominant change signal
- **Prescribed Fire pixels** (LCMS Change class 6)
- **Rangeland/Grassland land cover** — most-recent-year mosaic of Shrubs (7), Grass/Shrub Mix (8), and Grass/Forb/Herb (10)
- **Rangeland or Pasture land use** — LCMS Land Use class 5 for the most recent year
- **MTBS burn severity** time lapse

In [ ]:
Map.clearMap()

# ── Fire change masks (any year with that class = dominant change) ────────────
wildfire_any   = lcms.select('Change').map(lambda img: img.eq(7)).max().selfMask()
rx_fire_any    = lcms.select('Change').map(lambda img: img.eq(6)).max().selfMask()
any_fire       = wildfire_any.unmask(0).add(rx_fire_any.unmask(0)).gt(0).selfMask()

# ── Most-recent-year land cover and use ──────────────────────────────────────
recent_lc = lcms.filter(ee.Filter.eq('year', END_YEAR)).select('Land_Cover').mosaic()
recent_lu = lcms.filter(ee.Filter.eq('year', END_YEAR)).select('Land_Use').mosaic()

# Rangeland grass/shrub land cover (classes 7, 8, 10)
rangeland_lc = recent_lc.remap([7, 8, 10], [1, 1, 1], 0).selfMask()

# Rangeland or Pasture land use (class 5)
rangeland_lu = recent_lu.eq(5).selfMask()

# ── Fire intersected with rangeland land cover ────────────────────────────────
fire_in_rangeland = wildfire_any.updateMask(rangeland_lc)

# ── Add layers ────────────────────────────────────────────────────────────────
Map.addLayer(
    rangeland_lu,
    {'palette': ['a6976a'], 'opacity': 0.4},
    'Rangeland or Pasture Land Use (2024)',
    False,
)
Map.addLayer(
    rangeland_lc,
    {'palette': ['d4a843'], 'opacity': 0.5},
    'Grass/Shrub Land Cover (2024)',
    True,
)
Map.addLayer(
    rx_fire_any,
    {'palette': ['a10018'], 'opacity': 0.7},
    'Prescribed Fire — any year 1985-2024',
    True,
)
Map.addLayer(
    wildfire_any,
    {'palette': ['d54309'], 'opacity': 0.8},
    'Wildfire — any year 1985-2024',
    True,
)
Map.addLayer(
    fire_in_rangeland,
    {'palette': ['ff0000'], 'opacity': 0.9},
    'Wildfire IN Grass/Shrub Land Cover',
    True,
)

# Montana outline
Map.addLayer(
    montana.style(outlineWidth=2, fillColor='00000000', color='ffffff'),
    {},
    'Montana boundary',
    True,
)

Map.centerObject(aoi, 6)
Map.turnOnInspector()
Map.view()


## 2 · MTBS Burn Severity Time Lapse

Animate annual burn severity across Montana (1985–2024). Years with no burns show as unclassified. Watch how large fire years cluster in drought periods.

In [ ]:
Map.clearMap()

Map.addTimeLapse(
    mtbs,
    {'autoViz': True, 'canAreaChart': True},
    'MTBS Burn Severity 1985-2024',
    True,
)

# Overlay rangeland grass/shrub as static context layer
Map.addLayer(
    rangeland_lc,
    {'palette': ['d4a843'], 'opacity': 0.35},
    'Grass/Shrub Land Cover (2024)',
    True,
)
Map.addLayer(
    montana.style(outlineWidth=2, fillColor='00000000', color='ffffff'),
    {},
    'Montana boundary',
    True,
)

Map.centerObject(aoi, 6)
Map.view()


## 3 · Time-Series Charts

### 3a — Annual Fire Change Area (Wildfire vs Prescribed Fire)

Stacked line chart of how much of Montana shows wildfire vs. prescribed fire as the dominant change signal each year.

In [ ]:
fire_change_result = cl.summarize_and_chart(
    lcms.select('Change'),
    geometry=aoi,
    band_names='Change',
    scale=90,
    crs=CRS,
    area_format='Area (ha)',
    title='LCMS Change — Montana (1985-2024)',
    class_names_dict={
        6: 'Prescribed Fire',
        7: 'Wildfire',
    },
)
display(HTML(fire_change_result['chart_html']))


### 3b — Land Cover Trend in Montana

How the balance of rangeland/grassland vs. forest vs. other land cover has shifted over the full LCMS record.

In [ ]:
lc_result = cl.summarize_and_chart(
    lcms.select('Land_Cover'),
    geometry=aoi,
    band_names='Land_Cover',
    scale=90,
    crs=CRS,
    area_format='Percentage',
    title='Land Cover — Montana (1985-2024)',
    # Highlight rangeland classes; hide low-signal AK-only and mask classes
    class_names_dict={
        7:  'Shrubs',
        8:  'Grass/Forb/Herb & Shrubs Mix',
        10: 'Grass/Forb/Herb',
    },
)
display(HTML(lc_result['chart_html']))
